# Ajedrez online · Comparación entre cortes: log-loss, accuracy y matriz de confusión

**Proyecto Integrador · Ciencia de Datos, UTN FRM 2026**

**Pregunta.** El notebook 04 mostró que el modelo mejora sobre el piso a medida que avanza la partida. Este notebook compara **los cortes entre sí** con más detalle: ¿cuánto mejor se predice en cada momento, y esa mejora es real o es ruido, o es sólo que en los cortes tardíos quedan otras partidas?

**Qué hay en este notebook.**

1. el planteo;
2. los datos y las predicciones del modelo guardado, verificadas contra el notebook 04;
3. log-loss y accuracy, lado a lado, en cada corte;
4. la misma comparación sobre **población común** (las mismas partidas en todos los cortes);
5. intervalos de confianza y si el salto entre dos cortes es distinto de cero;
6. las matrices de confusión por tramo, una al lado de la otra, leídas como **recall** y como **precisión**;
7. la verificación;
8. las conclusiones.

**Qué no hay.** Ninguna decisión de modelado. El modelo es el que dejó congelado el notebook 04 (`models/modelo_en_vivo.joblib`) y acá sólo se lo mide sobre el test.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.app_data import REPORTE_MODELADO, leer_reporte
from src.build_app import curva_de_test, filas_de_test, por_tramo, predecir_test, raw_paths, verificar_contra_notebook
from src.comparacion_cortes import (
    decision, diferencias_entre_cortes, intervalos, metricas_por_corte, perdida_por_fila, poblacion_comun,
    replicas_bootstrap,
)
from src.metricas_tramo import TRAMOS, confusion_por_tramo, tramo_de_corte
from src.predict import MODEL_PATH, RESULT_ORDER, cargar_modelo
from src.utils import load_config

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
RANDOM_STATE = config["processing"]["random_state"]
REFERENCE_CUTS = config["live_prediction"]["cortes_referencia"]
# Same common population as notebook 04: the games still in play at ply 60 (~ median length).
COMMON_PLY = 60
N_REPLICAS = 1000
NOTEBOOK_START = time.perf_counter()

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
COLOR_MODEL, COLOR_FLOOR, COLOR_COMMON = "#216653", "#52665c", "#d95640"

## 1 · Planteo

**Dos métricas, cada una con su pregunta.**

- **Log-loss** es la métrica declarada del proyecto: la app muestra probabilidades, y la log-loss mide cuánta probabilidad le dio el modelo a lo que pasó. Es la que decidió el modelo en el notebook 04.
- **Accuracy** es la decisión fija: el modelo dice la clase más probable, ¿acertó? No decide nada, pero es lo que cualquiera entiende al mirar la app, y puede contar una historia distinta. Una probabilidad que pasa de 51 % a 70 % mejora mucho la log-loss y no cambia la accuracy; una que pasa de 49 % a 51 % casi no mueve la log-loss y cambia el acierto.

Las dos se informan siempre con su piso: `DummyClassifier(strategy="prior")` en log-loss y "siempre la clase más frecuente de train" en accuracy.

**Dos trampas al comparar cortes, y cómo se evitan.**

- **Supervivencia.** El ply 90 sólo tiene las partidas que llegaron al ply 90, que son largas, con más tablas y menos victorias blancas. Una mejora en ese corte puede ser del modelo o de la población. La **población común** (las partidas que llegan al ply 60, medidas en todos sus cortes hasta el 60) separa las dos cosas: ahí cada corte mide exactamente las mismas partidas.
- **Ruido.** Una diferencia entre dos cortes puede ser chica frente a lo que varía la métrica de una muestra a otra. Es la regla de la Unidad 3: una diferencia menor que su barra de error no es una diferencia. Como acá no hay pedazos de validación cruzada (el modelo ya está entrenado, se mide el test), la barra sale de un **bootstrap por partida**: se sortean partidas con reposición, mil veces, y se recalcula todo. Se sortean partidas y no filas porque los cortes de una misma partida no son independientes. El mismo sorteo vale para todos los cortes, así el cambio entre dos cortes también tiene su intervalo, pareado.

**Por qué tramos para la matriz.** Una matriz de confusión necesita bastantes filas para que sus porcentajes signifiquen algo, sobre todo en la columna de tablas. Se usan los seis tramos del notebook 04 (`src/metricas_tramo.py`), los mismos de la app.

## 2 · Datos y predicciones del modelo guardado

Las filas de train y test se arman con las mismas funciones que usa `python -m src.build_app` (`filas_de_test`, `predecir_test`), que a su vez reproducen la partición y los historiales del notebook 04. Antes de medir nada se verifica que el test reconstruido dé **lo mismo que el notebook 04**: la log-loss global, la de cada corte y la accuracy y la matriz de cada tramo. Si no coincide, el notebook falla: los números de acá tienen que poder ponerse al lado de los del 04.

Hace falta haber corrido antes el notebook 04, que deja `models/modelo_en_vivo.joblib` y `models/reporte_modelado.json`.

In [ ]:
games = pd.read_parquet(PROJECT_ROOT / config["paths"]["clean_parquet"])
cuts = pd.read_parquet(PROJECT_ROOT / config["paths"]["cortes_parquet"])
artifact = cargar_modelo(PROJECT_ROOT / MODEL_PATH)
report = leer_reporte(PROJECT_ROOT / REPORTE_MODELADO)
CLASE_PISO = report["clase_piso"]

started = time.perf_counter()
train, test = filas_de_test(games, cuts, raw_paths(config), artifact)
pred = predecir_test(train, test, artifact)
print(f"Test: {pred['GameUrl'].nunique():,} partidas, {len(pred):,} filas (partida, corte); "
      f"predicho en {time.perf_counter() - started:.0f} s")

# Same numbers as notebook 04, or stop here.
test_metrics = verificar_contra_notebook(pred, curva_de_test(pred), report)
band_metrics, _ = por_tramo(pred, report)
print(f"Coincide con el notebook 04: log-loss {test_metrics['log_loss']:.4f} contra {test_metrics['log_loss piso']:.4f} del piso; "
      f"accuracy y matriz de los {len(TRAMOS)} tramos iguales")
print(f"Piso en decisión fija: siempre «{CLASE_PISO}»")

## 3 · Log-loss y accuracy, lado a lado

Las dos métricas en los 25 cortes, cada una contra su piso, con su intervalo de 95 % por bootstrap. Abajo, la ganancia sobre el piso en la unidad de cada métrica: el porcentaje de log-loss que se ahorra y los puntos de accuracy que se suman.

Qué mirar: si las dos curvas de ganancia suben al mismo ritmo, o si hay momentos en que una se mueve y la otra no. En los tramos tardíos el piso empeora solo (quedan más tablas), así que una parte de la ganancia viene del piso y no del modelo; eso lo separa la sección 4.

In [ ]:
started = time.perf_counter()
by_cut = metricas_por_corte(pred, CLASE_PISO)
replicas_all = replicas_bootstrap(pred, CLASE_PISO, n_replicas=N_REPLICAS, random_state=RANDOM_STATE)
ci_all = intervalos(replicas_all, by_cut)
print(f"Bootstrap de {N_REPLICAS} réplicas sobre {pred['GameUrl'].nunique():,} partidas en {time.perf_counter() - started:.0f} s")


def band(ci, metric):
    return ci[ci["metrica"] == metric].set_index("corte_ply").sort_index()


def plot_side_by_side(ci, title, color=COLOR_MODEL):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True)
    panels = [
        (axes[0, 0], [("log_loss", "Modelo", color), ("log_loss piso", "Piso", COLOR_FLOOR)], "Log-loss (menos es mejor)", 1),
        (axes[0, 1], [("accuracy", "Modelo", color), ("accuracy piso", "Piso", COLOR_FLOOR)], "Accuracy (%)", 100),
        (axes[1, 0], [("mejora log_loss", "Modelo contra piso", color)], "Log-loss ahorrada sobre el piso (%)", 100),
        (axes[1, 1], [("puntos sobre piso", "Modelo contra piso", color)], "Puntos de accuracy sobre el piso", 1),
    ]
    for ax, series, ylabel, scale in panels:
        for metric, label, c in series:
            b = band(ci, metric)
            ax.plot(b.index, b["valor"] * scale, marker="o", markersize=3, color=c, label=label)
            ax.fill_between(b.index, b["bajo"] * scale, b["alto"] * scale, color=c, alpha=0.2, linewidth=0)
        ax.set_ylabel(ylabel)
        ax.legend(loc="best", fontsize=9)
    for ax in axes[1]:
        ax.axhline(0, color="#9e9e9e", linewidth=1)
        ax.set_xlabel("Plies jugados (corte)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_side_by_side(ci_all, "Todas las partidas que siguen en juego en cada corte (bandas: IC 95 % por bootstrap de partidas)")

wide = ci_all.pivot(index="corte_ply", columns="metrica", values=["valor", "bajo", "alto"])
table = pd.DataFrame({
    "partidas": by_cut.set_index("corte_ply")["partidas"],
    "log_loss": wide[("valor", "log_loss")], "log_loss piso": wide[("valor", "log_loss piso")],
    "mejora log_loss %": 100 * wide[("valor", "mejora log_loss")],
    "IC mejora %": [f"[{100 * lo:.1f}, {100 * hi:.1f}]" for lo, hi in zip(wide[("bajo", "mejora log_loss")], wide[("alto", "mejora log_loss")])],
    "accuracy %": 100 * wide[("valor", "accuracy")],
    "IC accuracy %": [f"[{100 * lo:.1f}, {100 * hi:.1f}]" for lo, hi in zip(wide[("bajo", "accuracy")], wide[("alto", "accuracy")])],
    "accuracy piso %": 100 * wide[("valor", "accuracy piso")],
    "puntos sobre piso": wide[("valor", "puntos sobre piso")],
    "IC puntos": [f"[{lo:.1f}, {hi:.1f}]" for lo, hi in zip(wide[("bajo", "puntos sobre piso")], wide[("alto", "puntos sobre piso")])],
})
display(table)

In [ ]:
# Where the two metrics tell different stories: rank of each cut by its gain over the floor.
gains = by_cut.set_index("corte_ply")[["mejora log_loss", "puntos sobre piso"]]
ranks = gains.rank(ascending=False).astype(int).rename(columns=lambda c: f"puesto {c}")
display(pd.concat([gains.assign(**{"mejora log_loss": 100 * gains["mejora log_loss"]}), ranks], axis=1))
rho = gains.corr(method="spearman").iloc[0, 1]
print(f"Correlación de Spearman entre las dos ganancias a lo largo de los cortes: {rho:.3f}")

opening = gains.loc[[c for c in gains.index if 5 <= c <= 20]]
print(f"Apertura (plies 5–20): la log-loss ahorrada va de {100 * opening['mejora log_loss'].min():.1f} % a "
      f"{100 * opening['mejora log_loss'].max():.1f} %; los puntos de accuracy, de {opening['puntos sobre piso'].min():.1f} a "
      f"{opening['puntos sobre piso'].max():.1f}")

**Lectura.** _Pendiente: completar con los números de la corrida._ Puntos a cubrir:

- antes de mover, cuánto ahorra el modelo en log-loss y cuántos puntos de accuracy suma, y si el intervalo de los puntos sobre el piso excluye el cero;
- si en la apertura (plies 5–20) las dos ganancias se mueven juntas o una se estanca;
- desde qué corte las dos curvas crecen claramente, y cómo cambia el ancho de los intervalos al final (quedan menos partidas).

## 4 · Población común: las mismas partidas en todos los cortes

Las partidas de test que siguen en juego en el ply 60, medidas en cada corte del 0 al 60. Es la misma población común del notebook 04, que ahí se usó sólo para la log-loss; acá se agrega la accuracy y los intervalos.

Si la curva de la población común sube igual que la de todas las partidas, la mejora con el avance es del modelo. Si sube menos, una parte de lo que parecía mejora era que cambiaron las partidas.

In [ ]:
common = poblacion_comun(pred, COMMON_PLY)
by_cut_common = metricas_por_corte(common, CLASE_PISO)
replicas_common = replicas_bootstrap(common, CLASE_PISO, n_replicas=N_REPLICAS, random_state=RANDOM_STATE)
ci_common = intervalos(replicas_common, by_cut_common)
print(f"Población común: {common['GameUrl'].nunique():,} partidas que llegan al ply {COMMON_PLY}, "
      f"en {common['corte_ply'].nunique()} cortes ({len(common):,} filas)")

# Notebook 04 already measured this population's log-loss: same numbers, or stop.
by_population = pd.DataFrame(report["test_por_corte"])
nb04_common = by_population[by_population["población"] == f"llegan al ply {COMMON_PLY}"].set_index("corte_ply")
assert len(nb04_common), "El reporte del 04 no trae la población común"
mine = by_cut_common.set_index("corte_ply").reindex(nb04_common.index)
for metric in ["log_loss", "log_loss piso", "accuracy", "accuracy piso"]:
    gap = (mine[metric] - nb04_common[metric]).abs().max()
    assert gap < 1e-6, f"Población común: {metric} difiere del notebook 04 en {gap:.2e}"
print(f"Coincide con la población común del notebook 04 en {len(nb04_common)} cortes (log-loss y accuracy, modelo y piso)")

plot_side_by_side(ci_common, f"Población común: las mismas {common['GameUrl'].nunique():,} partidas en cada corte", color=COLOR_COMMON)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, ylabel in [(axes[0], "accuracy", "Accuracy (%)"), (axes[1], "puntos sobre piso", "Puntos de accuracy sobre el piso")]:
    scale = 100 if metric == "accuracy" else 1
    for ci, label, c in [(ci_all, "Todas las que siguen en juego", COLOR_MODEL), (ci_common, f"Población común (llegan al {COMMON_PLY})", COLOR_COMMON)]:
        b = band(ci, metric).loc[:COMMON_PLY]
        ax.plot(b.index, b["valor"] * scale, marker="o", markersize=3, color=c, label=label)
        ax.fill_between(b.index, b["bajo"] * scale, b["alto"] * scale, color=c, alpha=0.2, linewidth=0)
    ax.set(xlabel="Plies jugados (corte)", ylabel=ylabel)
    ax.legend(fontsize=9)
fig.suptitle("Accuracy: todas las partidas contra la población común")
plt.tight_layout()
plt.show()

compare = pd.DataFrame({
    "accuracy % (todas)": 100 * by_cut.set_index("corte_ply")["accuracy"],
    "accuracy % (común)": 100 * by_cut_common.set_index("corte_ply")["accuracy"],
    "accuracy piso % (común)": 100 * by_cut_common.set_index("corte_ply")["accuracy piso"],
    "mejora log_loss % (todas)": 100 * by_cut.set_index("corte_ply")["mejora log_loss"],
    "mejora log_loss % (común)": 100 * by_cut_common.set_index("corte_ply")["mejora log_loss"],
}).loc[REFERENCE_CUTS]
display(compare)

**Lectura.** _Pendiente: completar con los números de la corrida._ Puntos a cubrir:

- cuánto sube la accuracy de la población común del ply 0 al 60, contra lo que sube la de todas las partidas;
- si en el corte 0 la población común es más difícil (partidas largas, más parejas) y cuánto;
- qué parte de la mejora con el avance queda en pie cuando las partidas son las mismas.

## 5 · ¿El salto entre dos cortes es real?

Sobre la población común, el cambio de accuracy y de log-loss entre cortes de referencia consecutivos (`cortes_referencia` de `config.yaml`) y entre el principio y el ply 60. El intervalo es pareado: en cada réplica del bootstrap se miden las mismas partidas en los dos cortes. Un intervalo que no cruza el cero es un salto que no se explica por la muestra.

En accuracy, positivo es mejor; en log-loss, negativo es mejor.

In [ ]:
common_cuts = [c for c in REFERENCE_CUTS if c <= COMMON_PLY]
pairs = list(zip(common_cuts[:-1], common_cuts[1:])) + [(common_cuts[0], common_cuts[-1])]
jumps = diferencias_entre_cortes(replicas_common, by_cut_common, pairs)
shown = jumps.copy()
is_accuracy = shown["metrica"] == "accuracy"
shown.loc[is_accuracy, ["cambio", "bajo", "alto"]] *= 100  # accuracy in points
shown["unidad"] = np.where(is_accuracy, "puntos de accuracy", "log-loss")
display(shown.set_index(["metrica", "desde", "hasta"])[["unidad", "cambio", "bajo", "alto", "distinto de cero"]])

for _, row in shown[is_accuracy].iterrows():
    verdict = "salto real" if row["distinto de cero"] else "no se distingue del ruido"
    print(f"Accuracy del ply {row['desde']} al {row['hasta']}: {row['cambio']:+.1f} puntos [{row['bajo']:+.1f}, {row['alto']:+.1f}] → {verdict}")

**Lectura.** _Pendiente: completar con los números de la corrida._ Puntos a cubrir: qué saltos son reales y cuáles no, en particular en la apertura (0 → 10 → 20), y si la log-loss y la accuracy coinciden en qué saltos importan.

## 6 · Las matrices de confusión, tramo por tramo, una al lado de la otra

Seis matrices, una por tramo, leídas de dos formas:

- **Recall** (% de la fila): de cada resultado real, qué dijo el modelo. La diagonal dice qué parte de cada resultado reconoce.
- **Precisión** (% de la columna): de cada cosa que dijo el modelo, qué pasó. La diagonal dice cuánto creerle cuando dice algo. Una clase que el modelo nunca eligió en un tramo no tiene precisión: la celda queda en "—" (no es un 0 %).

Juntas, en la misma escala de color, muestran cómo cambian los errores con el avance. Después, la misma figura sobre la población común, que llega hasta el tramo 45–60.

In [ ]:
def plot_confusions(confusion, title):
    bands = [t for t in TRAMOS if t in set(confusion["tramo"])]
    fig, axes = plt.subplots(2, len(bands), figsize=(3.1 * len(bands), 7.2), squeeze=False)
    for row, (column, name) in enumerate([("porcentaje_fila", "Recall (% de la fila)"), ("porcentaje_columna", "Precisión (% de la columna)")]):
        for k, t in enumerate(bands):
            ax = axes[row, k]
            m = confusion[confusion["tramo"] == t].pivot(index="real", columns="predicho", values=column).reindex(index=RESULT_ORDER, columns=RESULT_ORDER)
            # aspect="auto" lets tight_layout make room for the tick labels between the two rows.
            ax.imshow(m.fillna(0).to_numpy(), cmap="Greens", vmin=0, vmax=100, aspect="auto")
            for i in range(3):
                for j in range(3):
                    v = m.iloc[i, j]
                    ax.text(j, i, "—" if pd.isna(v) else f"{v:.0f}", ha="center", va="center", fontsize=10,
                            color="white" if (not pd.isna(v) and v > 55) else "#173b32")
            ax.set_xticks(range(3), ["Blancas", "Tablas", "Negras"], fontsize=8)
            ax.set_yticks(range(3), ["Blancas", "Tablas", "Negras"] if k == 0 else [""] * 3, fontsize=8)
            ax.grid(False)
            if row == 0:
                ax.set_title(f"Plies {t}", fontsize=10)
            if k == 0:
                ax.set_ylabel(f"{name}\nreal", fontsize=9)
            if row == 1:
                ax.set_xlabel("predicho", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def diagonal(confusion):
    d = confusion[confusion["real"] == confusion["predicho"]]
    out = d.pivot(index="tramo", columns="real", values=["porcentaje_fila", "porcentaje_columna"])
    out.columns = [f"{'recall' if a == 'porcentaje_fila' else 'precisión'} {b}" for a, b in out.columns]
    counts = confusion.groupby(["tramo", "predicho"])["filas"].sum().unstack()
    out["veces que dijo tablas"] = counts["Empate"]
    return out.loc[[t for t in TRAMOS if t in out.index]]


confusion_all = confusion_por_tramo(pred["resultado"], decision(pred), pred["corte_ply"])
plot_confusions(confusion_all, "Matriz de confusión por tramo · todas las partidas que siguen en juego")
display(diagonal(confusion_all))

confusion_common = confusion_por_tramo(common["resultado"], decision(common), common["corte_ply"])
plot_confusions(confusion_common, f"Matriz de confusión por tramo · población común (llegan al ply {COMMON_PLY})")
display(diagonal(confusion_common))

**Lectura.** _Pendiente: completar con los números de la corrida._ Puntos a cubrir:

- **recall:** la asimetría blancas/negras antes de mover y cómo se empareja al final;
- **precisión:** cuánto creerle al modelo cuando dice "ganan blancas" o "ganan negras" en cada tramo, y si sube al mismo ritmo que el recall;
- **tablas:** desde qué tramo las elige, cuántas veces y con qué precisión (rara vez las dice; cuando las dice, ¿acierta?);
- **población común:** si los cambios de la matriz se sostienen con las mismas partidas.

## 7 · Verificación

Los controles que sostienen las tablas de arriba:

- el test reconstruido coincide con el notebook 04 (ya se comprobó en la sección 2 y, para la población común, en la sección 4);
- la log-loss por fila promedia lo mismo que `sklearn.metrics.log_loss`;
- cada corte de la población común tiene las mismas partidas;
- las filas de cada matriz suman 100 % y las columnas con predicciones también;
- el intervalo de cada accuracy contiene su valor de muestra.

In [ ]:
from sklearn.metrics import accuracy_score, log_loss

CLASSES = sorted(RESULT_ORDER)
proba = pred[["p_empate", "p_blancas", "p_negras"]].to_numpy()
assert abs(perdida_por_fila(pred).mean() - log_loss(pred["resultado"], proba, labels=CLASSES)) < 1e-9
assert abs(by_cut["filas"].sum() - len(pred)) == 0
assert np.isclose(accuracy_score(pred["resultado"], decision(pred)), report["test"]["accuracy"], atol=1e-9)

assert common.groupby("corte_ply")["GameUrl"].apply(frozenset).nunique() == 1, "La población común cambia entre cortes"
assert set(common["corte_ply"]) == {c for c in pred["corte_ply"].unique() if c <= COMMON_PLY}

for confusion in (confusion_all, confusion_common):
    rows = confusion.groupby(["tramo", "real"])["porcentaje_fila"].sum()
    cols = confusion.groupby(["tramo", "predicho"])["porcentaje_columna"].sum(min_count=1).dropna()
    assert np.allclose(rows, 100) and np.allclose(cols, 100), "Una matriz no suma 100 %"
    never = confusion.groupby(["tramo", "predicho"])["filas"].transform("sum") == 0
    assert confusion.loc[never, "porcentaje_columna"].isna().all(), "Precisión de una clase que nunca se predijo"
assert confusion_all["filas"].sum() == len(pred) and confusion_common["filas"].sum() == len(common)

for ci in (ci_all, ci_common):
    acc = band(ci, "accuracy")
    assert ((acc["bajo"] <= acc["valor"]) & (acc["valor"] <= acc["alto"])).all()
assert (tramo_de_corte(pred["corte_ply"]) != "").all()

print(f"Verificación OK. Notebook completo en {(time.perf_counter() - NOTEBOOK_START) / 60:.1f} min")

## 8 · Conclusiones

_Pendiente: completar con los números de la corrida._ Estructura prevista:

- **Las dos métricas, corte por corte.** Dónde coinciden y dónde no la log-loss y la accuracy.
- **Mejora del modelo o cambio de partidas.** Lo que dice la población común.
- **Saltos reales.** Qué cambios entre cortes superan su intervalo.
- **Errores.** Cómo cambian recall y precisión con el avance, y qué pasa con las tablas.